# Day 7 Block 4 — End-to-End Stress Test

25 queries across 6 categories. Captures score, answer, sources, latency.
Validates the system end-to-end and produces evaluation data for the README.

Categories:
- in-domain-oncology (5)
- in-domain-cardio-metabolic (4)
- in-domain-specific-edge (4) — sneaky cases that should refuse at Layer 2
- out-of-domain-clearly (4) — bread, code, hiking
- adversarial-medical (4) — medical-sounding but answer not in corpus
- control-easy (4) — should get clean cited answers

In [1]:
import sys, os, time
sys.path.insert(0, "..")  # so we can import from src/

from src.rag import (
    load_chunks,
    load_index,
    load_model,
    make_groq_client,
    generate,
)

# Load resources once for the whole notebook
print("Loading resources...")
t0 = time.time()
model = load_model()
index = load_index()
chunks = load_chunks()
client = make_groq_client()
print(f"Loaded in {time.time() - t0:.1f}s")
print(f"  Chunks:  {len(chunks):,}")
print(f"  Index:   {index.ntotal:,} vectors, {index.d} dims")

Loading resources...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded in 2.6s
  Chunks:  3,264
  Index:   3,264 vectors, 768 dims


In [ ]:
#Step 2: Define the 25-query test set
# 25 queries across 6 categories.
# (category, query, expected_behavior)
TEST_QUERIES = [
    # ---- 1. Control: easy in-domain queries that should answer cleanly ----
    ("control-easy", "What trials are studying BRAF mutations in melanoma?", "cited_answer"),
    ("control-easy", "What trials test pembrolizumab for non-small cell lung cancer?", "cited_answer"),
    ("control-easy", "Which studies investigate HER2 positive breast cancer?", "cited_answer"),
    ("control-easy", "What trials look at heart failure with reduced ejection fraction?", "cited_answer"),

    # ---- 2. In-domain oncology, more specific ----
    ("in-domain-oncology", "Trials combining immunotherapy with chemotherapy in NSCLC", "cited_answer"),
    ("in-domain-oncology", "Studies of dabrafenib and trametinib combination therapy", "cited_answer"),
    ("in-domain-oncology", "Neoadjuvant immunotherapy trials in resectable melanoma", "cited_answer"),
    ("in-domain-oncology", "ctDNA biomarker monitoring in advanced cancer", "cited_answer"),
    ("in-domain-oncology", "Trials studying antibody-drug conjugates in breast cancer", "cited_answer"),

    # ---- 3. In-domain cardio/metabolic ----
    ("in-domain-cardio-metabolic", "SGLT2 inhibitor trials in type 2 diabetes", "cited_answer"),
    ("in-domain-cardio-metabolic", "GLP-1 agonist studies in obesity and diabetes", "cited_answer"),
    ("in-domain-cardio-metabolic", "Heart failure trials in patients with preserved ejection fraction", "cited_answer"),
    ("in-domain-cardio-metabolic", "What trials are studying anticoagulation in atrial fibrillation?", "cited_answer"),

    # ---- 4. Sneaky edge: medical, but specific enough corpus may not answer ----
    ("in-domain-specific-edge", "Recommended insulin dosage for pregnant women with type 1 diabetes", "should_refuse"),
    ("in-domain-specific-edge", "What is the optimal radiation dose for early-stage prostate cancer?", "should_refuse"),
    ("in-domain-specific-edge", "Drug interactions between metformin and contrast dye", "should_refuse"),
    ("in-domain-specific-edge", "Long-term cognitive effects of pediatric chemotherapy", "should_refuse"),

    # ---- 5. Out-of-domain (clearly non-medical) ----
    ("out-of-domain-clearly", "How do I bake sourdough bread?", "should_refuse"),
    ("out-of-domain-clearly", "Python list comprehension syntax", "should_refuse"),
    ("out-of-domain-clearly", "Best hiking trails in New Hampshire", "should_refuse"),
    ("out-of-domain-clearly", "How to fix a leaking kitchen faucet", "should_refuse"),

    # ---- 6. Adversarial: medical-sounding, but specific answer is not in corpus ----
    ("adversarial-medical", "What is the LD50 of acetaminophen in adults?", "should_refuse"),
    ("adversarial-medical", "Side effects of taking ibuprofen daily for 10 years", "should_refuse"),
    ("adversarial-medical", "Why does my left arm hurt after a flu shot?", "should_refuse"),
    ("adversarial-medical", "How quickly does penicillin clear a strep throat infection?", "should_refuse"),
]

print(f"Total test queries: {len(TEST_QUERIES)}")
from collections import Counter
cat_counts = Counter(c for c, _, _ in TEST_QUERIES)
for cat, n in cat_counts.items():
    print(f"  {cat:<28} {n}")

Total test queries: 25
  control-easy                 4
  in-domain-oncology           5
  in-domain-cardio-metabolic   4
  in-domain-specific-edge      4
  out-of-domain-clearly        4
  adversarial-medical          4


In [3]:
#Step 3: Run the stress test
import time

results_log = []

for i, (category, query, expected) in enumerate(TEST_QUERIES, 1):
    print(f"[{i:2d}/{len(TEST_QUERIES)}] {category:<28} {query[:55]}")
    t0 = time.time()
    try:
        result = generate(query, client, model, index, chunks)
        latency_s = time.time() - t0
        error = None
    except Exception as e:
        result = {"answer": "", "sources": [], "top_score": 0.0, "refused_at": "ERROR"}
        latency_s = time.time() - t0
        error = str(e)
        print(f"    ERROR: {e}")

    # Classify outcome by reading the answer (matches what app.py does)
    answer = result["answer"]
    if result["refused_at"] == "threshold":
        outcome = "refused_layer1"
    elif "outside the scope" in answer:
        outcome = "refused_layer2_scope"
    elif "don't have enough information" in answer:
        outcome = "refused_layer2_no_info"
    elif result["refused_at"] == "ERROR":
        outcome = "ERROR"
    else:
        outcome = "answered"

    results_log.append({
        "id": i,
        "category": category,
        "query": query,
        "expected": expected,
        "outcome": outcome,
        "top_score": round(result["top_score"], 4),
        "latency_s": round(latency_s, 2),
        "n_sources": len(result["sources"]),
        "source_nct_ids": ",".join(s["nct_id"] for s in result["sources"]),
        "answer": answer,
        "error": error,
    })

    # Pace ourselves so Groq doesn't rate-limit us
    time.sleep(0.5)

print(f"\nFinished. Logged {len(results_log)} results.")

[ 1/25] control-easy                 What trials are studying BRAF mutations in melanoma?
[ 2/25] control-easy                 What trials test pembrolizumab for non-small cell lung 
[ 3/25] control-easy                 Which studies investigate HER2 positive breast cancer?
[ 4/25] control-easy                 What trials look at heart failure with reduced ejection
[ 5/25] in-domain-oncology           Trials combining immunotherapy with chemotherapy in NSC
[ 6/25] in-domain-oncology           Studies of dabrafenib and trametinib combination therap
[ 7/25] in-domain-oncology           Neoadjuvant immunotherapy trials in resectable melanoma
[ 8/25] in-domain-oncology           ctDNA biomarker monitoring in advanced cancer
[ 9/25] in-domain-oncology           Trials studying antibody-drug conjugates in breast canc
[10/25] in-domain-cardio-metabolic   SGLT2 inhibitor trials in type 2 diabetes
[11/25] in-domain-cardio-metabolic   GLP-1 agonist studies in obesity and diabetes
[12/25] in-doma

In [4]:
#Step 4: Summary table + correctness check
import pandas as pd

df = pd.DataFrame(results_log)

# ---- Quick stats ----
print(f"Total queries:     {len(df)}")
print(f"Total errors:      {(df['outcome'] == 'ERROR').sum()}")
print(f"Avg latency:       {df['latency_s'].mean():.2f}s")
print(f"P50 / P95 latency: {df['latency_s'].quantile(0.5):.2f}s / {df['latency_s'].quantile(0.95):.2f}s")
print(f"Score range:       {df['top_score'].min():.3f} – {df['top_score'].max():.3f}")

print("\n" + "=" * 70)
print("OUTCOMES BY CATEGORY")
print("=" * 70)
outcome_table = pd.crosstab(df["category"], df["outcome"], margins=True, margins_name="TOTAL")
print(outcome_table)

print("\n" + "=" * 70)
print("EXPECTATIONS vs OUTCOMES")
print("=" * 70)
df["correct"] = df.apply(
    lambda r: (r["expected"] == "cited_answer" and r["outcome"] == "answered")
              or (r["expected"] == "should_refuse" and r["outcome"].startswith("refused")),
    axis=1,
)
print(f"Correct:    {df['correct'].sum()} / {len(df)}")
print(f"Incorrect:  {(~df['correct']).sum()} / {len(df)}")

if (~df["correct"]).any():
    print("\nMISCLASSIFIED — these need manual review:")
    for _, row in df[~df["correct"]].iterrows():
        print(f"  [{row['id']:2d}] expected={row['expected']:<14} got={row['outcome']:<24} score={row['top_score']:.3f}")
        print(f"       Q: {row['query']}")
        print(f"       A: {row['answer'][:200]}")
        print()

Total queries:     25
Total errors:      0
Avg latency:       4.20s
P50 / P95 latency: 4.52s / 7.89s
Score range:       0.844 – 0.960

OUTCOMES BY CATEGORY
outcome                     answered  refused_layer2_no_info  \
category                                                       
adversarial-medical                0                       1   
control-easy                       3                       1   
in-domain-cardio-metabolic         4                       0   
in-domain-oncology                 3                       2   
in-domain-specific-edge            0                       3   
out-of-domain-clearly              0                       0   
TOTAL                             10                       7   

outcome                     refused_layer2_scope  TOTAL  
category                                                 
adversarial-medical                            3      4  
control-easy                                   0      4  
in-domain-cardio-metabolic         

In [5]:
#Step 5: Save the CSV + investigate query
# ---- Save the CSV for Day 8 RAGAS evaluation + README ----
output_path = "../data/eval_results.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")

# ---- Investigate query #9 (antibody-drug conjugates) ----
# Either the corpus lacks ADC trials, or retrieval missed them.
# Let's look at what FAISS retrieved.
print("\n" + "=" * 70)
print("INVESTIGATION: Query #9 retrieved sources")
print("=" * 70)

q9 = df[df["id"] == 9].iloc[0]
print(f"Query:      {q9['query']}")
print(f"Top score:  {q9['top_score']}")
print(f"Sources:    {q9['source_nct_ids']}")
print()

# Pull the actual retrieved chunks
from src.rag import retrieve
results = retrieve(q9["query"], model, index, chunks, k=5)
for r in results:
    print(f"  {r['nct_id']} (score={r['score']:.3f}) — {r['title'][:90]}")

# ---- Bonus check: does the corpus contain "antibody-drug conjugate" or "ADC" anywhere? ----
print("\n" + "=" * 70)
print("CORPUS CHECK: how many chunks mention antibody-drug conjugates?")
print("=" * 70)
keywords = ["antibody-drug conjugate", "antibody drug conjugate", "ADC ", "trastuzumab deruxtecan", "T-DXd", "sacituzumab"]
for kw in keywords:
    count = sum(1 for c in chunks if kw.lower() in c["text"].lower())
    print(f"  {kw!r:<35} appears in {count} chunks")

Saved 25 rows to ../data/eval_results.csv

INVESTIGATION: Query #9 retrieved sources
Query:      Trials studying antibody-drug conjugates in breast cancer
Top score:  0.9327
Sources:    NCT00004888,NCT06685796,NCT04059003,NCT03934905,NCT05479409

  NCT00004888 (score=0.933) — Combination Chemotherapy With or Without Trastuzumab in Treating Women With Metastatic Bre
  NCT06685796 (score=0.924) — A Study of BEBT-209 in Combination With Chemotherapy for the Treatment of Advanced Triple-
  NCT04059003 (score=0.922) — CTC Changes and Efficacy of Neoadjuvant Chemotherapy for Triple-negative Breast Cancer
  NCT03934905 (score=0.922) — Protective Effects of the Nutritional Supplement Sulforaphane on Doxorubicin-Associated Ca
  NCT05479409 (score=0.921) — Neoadjuvant Radiation in Locally Advanced Breast Cancer

CORPUS CHECK: how many chunks mention antibody-drug conjugates?
  'antibody-drug conjugate'           appears in 1 chunks
  'antibody drug conjugate'           appears in 0 chunks
  'ADC